In [1]:
# inlegalbert_dual_kg_rag_daiscl_ppo.py
#
# ══════════════════════════════════════════════════════════════════════════════
# UNIFIED PIPELINE:
#   InLegalBERT + BiLSTM + MHA + CRF
#   + Dual KG-RAG (G_all + G_min, rare-node amplification)
#   + Adaptive Confusion Cross-Edges
#   + Discourse-Aware Imbalance-Aware SCL  (DA-IA-SCL)
#   + Rare-only Intra-class Manifold Mixup
#   + PPO Reinforcement Learning  (Phase C+D)
#
# ──────────────────────────────────────────────────────────────────────────────
# KEY DESIGN PRINCIPLE: SCL and PPO must NOT interfere.
#
#   Problem: If PPO's reward shaping or policy gradient updates also flow
#   through the SCL projection head, the contrastive clusters become unstable,
#   because PPO optimises a different objective (reward maximisation) that does
#   not respect the metric-learning geometry built by SCL.
#
#   Solution — strict architectural separation:
#
#     SCL path  :  sent_vecs (256-d) → scl_proj (MLP) → L2-norm → contrastive loss
#                  ↑ trained only in Phase A (base) and Phase B (KG fine-tune)
#                  ↑ FROZEN during PPO phases C+D
#
#     PPO path  :  fused_ctx (128-d) → ActorHead / CriticHead → policy / value
#                  ↑ trained only in Phase C+D
#                  ↑ never backpropagates into scl_proj or the contrastive loss
#
#   The SCL-trained encoder produces richer, more separable sent_vecs that the
#   PPO actor inherits via the frozen base.  PPO then specialises the POLICY
#   HEAD on top of these high-quality features — pure additive improvement.
#
#   Additionally, PPO's KG-disagreement and curriculum rewards are computed
#   using the same sent_vecs (256-d) that SCL shaped, so PPO benefits from
#   the cleaner embedding space without any gradient coupling.
#
# ──────────────────────────────────────────────────────────────────────────────
# PHASE A   ─ Base model: InLegalBERT + BiLSTM + MHA + CRF + DA-IA-SCL + Mixup
# PHASE A→B ─ Confusion analysis, adaptive weights, Dual KG build
# PHASE B   ─ Dual KG-RAG fine-tuning + DA-IA-SCL on fused embeddings + Mixup
# PHASE C+D ─ PPO RL (SCL projection head FROZEN, policy/value heads trained)
# ══════════════════════════════════════════════════════════════════════════════

import os, json, random, time, math
from collections import Counter, defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH     = "dataset/build_train.jsonl"
DEV_PATH       = "dataset/build_dev.jsonl"
TEST_PATH      = "dataset/build_test.jsonl"
OUT_DIR        = "rrc_daiscl_ppo_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR,        exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# ── General ────────────────────────────────────────────────
SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS          = 8
BERT_LR_DECAY               = 0.9
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_RATIO                = 0.05
RARE_THRESHOLD              = 0.05
ES_PATIENCE                 = 10
ES_MIN_DELTA                = 1e-4

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT   = 0.2
LABEL_SMOOTHING = 0.1

# ── Phase A ─────────────────────────────────────────────────
NUM_EPOCHS_BASE = 60

# ── Phase B (Dual KG-RAG) ───────────────────────────────────
NUM_EPOCHS_KG  = 20
ES_PATIENCE_KG = 5

KG_TOP_K           = 3
KG_TOP_NODES       = 5
KG_MIN_NODES       = 3
KG_HOP             = 1
UNCERTAINTY_THRESH = 0.7
RARE_ALWAYS_KG     = True
KG_FUSION_DIM      = 256

RARITY_BETA        = 0.3
ALPHA_LOW_ENTROPY  = 0.7
ALPHA_HIGH_ENTROPY = 0.4

RST_INTRA_THRESH = 0.6
RST_CROSS_THRESH = 0.5

# ── Adaptive Confusion Edges ────────────────────────────────
CONF_BASE_ALPHA            = 0.40
CONF_EDGE_WEIGHT_DEFAULT   = 0.90
CONF_TOP_K                 = 3
CONF_SIM_FLOOR             = 0.30
CONF_MAX_PAIRS_PER_CLASS   = 30

# ── Rare-node amplification ─────────────────────────────────
RARE_AMP_FACTOR = 5
RARE_AMP_NOISE  = 0.008

# ── DA-IA-SCL ───────────────────────────────────────────────
SCL_WEIGHT            = 0.45
SCL_TEMPERATURE       = 0.07
SCL_BASE_TEMPERATURE  = 0.07
SCL_MINORITY_BOOST    = 3.0
SCL_MIN_POSITIVES     = 1
SCL_SAMPLES_PER_CLASS = 6
SCL_PROJ_DIM          = 128
SCL_IMBAL_GAMMA       = 0.5
DISC_POS_SCALE        = 1.5
DISC_NEG_SCALE        = 1.5
DISC_COMPAT_THRESH    = 0.3

# ── Rare-only Mixup ─────────────────────────────────────────
MIXUP_ALPHA             = 0.40
MIXUP_LOSS_WEIGHT       = 0.30
MIXUP_MIN_RARE_IN_BATCH = 2
MIXUP_NOISE_STD         = 0.01

# ── PPO (Phase C+D) ─────────────────────────────────────────
REWARD_RARE_CORRECT =  2.0
REWARD_RARE_WRONG   = -3.0
REWARD_MAJ_CORRECT  =  0.3
REWARD_MAJ_WRONG    =  0.0

PPO_EPOCHS       = 20
PPO_MINI_EPOCHS  = 4
PPO_CLIP_EPS     = 0.2
PPO_KL_COEF      = 0.1
PPO_VALUE_COEF   = 0.5
PPO_ENTROPY_COEF = 0.01
PPO_LR           = 3e-5
PPO_GRAD_CLIP    = 1.0
GAE_GAMMA        = 0.99
GAE_LAMBDA       = 0.95
ROLLOUT_DOCS     = 16
ES_PATIENCE_PPO  = 7

PHASE_D_MODE     = "combined"   # "curriculum" | "self_play" | "kg_disagree" | "combined"

CORRECTION_LR        = 1e-4
CORRECTION_EPOCHS    = 15
CORRECTION_NEG_RATIO = 3

CURRICULUM_WINDOW = 5
CURRICULUM_MIN_W  = 0.5
CURRICULUM_MAX_W  = 3.0

KG_DISAGREE_BONUS = 0.8

# ── Labels ─────────────────────────────────────────────────
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L, B  = batch[0]["input_ids"].shape[1], len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), -100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# DISCOURSE TRANSITION MATRIX
# ═══════════════════════════════════════════════════════════
def build_discourse_transition_matrix(docs: list) -> torch.Tensor:
    count_mat = np.zeros((NUM_LABELS, NUM_LABELS), dtype=np.float64)
    for sents, labs in docs:
        for k in range(len(labs) - 1):
            i, j = labs[k], labs[k + 1]
            if 0 <= i < NUM_LABELS and 0 <= j < NUM_LABELS:
                count_mat[i, j] += 1.0
    row_sums = count_mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    T = (count_mat / row_sums).astype(np.float32)
    print("\n📐 Discourse Transition Matrix (top-3 successors per class):")
    for i, lbl in enumerate(LABELS):
        top3 = np.argsort(-T[i])[:3]
        vals = [(id2label[j], f"{T[i,j]:.3f}") for j in top3 if T[i, j] > 0.0]
        print(f"   {lbl:<20} → {vals}")
    return torch.tensor(T, dtype=torch.float32)


# ═══════════════════════════════════════════════════════════
# SCL CLASS WEIGHTS
# ═══════════════════════════════════════════════════════════
def compute_scl_class_weights(label_freqs: dict, rare_ids: list,
                               gamma: float = SCL_IMBAL_GAMMA) -> torch.Tensor:
    weights = []
    for i in range(NUM_LABELS):
        freq = max(label_freqs.get(id2label[i], 1e-6), 1e-6)
        weights.append(freq ** (-gamma))
    weights = torch.tensor(weights, dtype=torch.float32)
    weights = weights / weights.mean()
    print(f"\n⚖️  SCL class weights (inv-freq^{gamma}, mean-normalised):")
    for i, lbl in enumerate(LABELS):
        flag = " ← RARE" if label2id[lbl] in rare_ids else ""
        print(f"   {lbl:<20}  w={float(weights[i]):.3f}{flag}")
    return weights


# ═══════════════════════════════════════════════════════════
# ★ DISCOURSE-AWARE IMBALANCE-AWARE SUPERVISED CONTRASTIVE LOSS
# ═══════════════════════════════════════════════════════════
class DiscourseAwareSCLoss(nn.Module):
    """
    DA-IA-SCL:
      1. Per-anchor weight: w_i = class_weights[label_i] * minority_boost (if rare)
      2. Positive-pair boost: T[i,j] > disc_compat_thresh → * DISC_POS_SCALE
      3. Negative-pair penalty: T[i,j] <= disc_compat_thresh → * DISC_NEG_SCALE
    """
    def __init__(self,
                 temperature=SCL_TEMPERATURE,
                 base_temperature=SCL_BASE_TEMPERATURE,
                 minority_boost=SCL_MINORITY_BOOST,
                 disc_pos_scale=DISC_POS_SCALE,
                 disc_neg_scale=DISC_NEG_SCALE,
                 disc_compat_thresh=DISC_COMPAT_THRESH):
        super().__init__()
        self.temperature       = temperature
        self.base_temperature  = base_temperature
        self.minority_boost    = minority_boost
        self.disc_pos_scale    = disc_pos_scale
        self.disc_neg_scale    = disc_neg_scale
        self.disc_compat_thresh = disc_compat_thresh
        self.register_buffer("T", torch.zeros(NUM_LABELS, NUM_LABELS))

    def set_transition_matrix(self, T: torch.Tensor):
        self.T = T.to(self.T.device if hasattr(self, "T") else torch.device("cpu"))

    def forward(self, features: torch.Tensor, labels: torch.Tensor,
                class_weights: torch.Tensor, rare_ids: list = None) -> torch.Tensor:
        device = features.device
        N = features.shape[0]
        if N < 2:
            return torch.tensor(0.0, device=device, requires_grad=True)

        T_dev = self.T.to(device)

        # Anchor weights
        anchor_w = class_weights[labels].to(device)
        if rare_ids:
            rare_mask = torch.zeros(N, dtype=torch.bool, device=device)
            for rid in rare_ids:
                rare_mask |= (labels == rid)
            anchor_w = anchor_w.clone()
            anchor_w[rare_mask] *= self.minority_boost

        # Similarities
        sim = torch.mm(features, features.T) / self.temperature  # (N,N)

        lbl_row  = labels.unsqueeze(1)
        lbl_col  = labels.unsqueeze(0)
        pos_mask = (lbl_row == lbl_col).float()
        self_mask = torch.eye(N, device=device)
        pos_mask  = pos_mask - self_mask

        disc_w       = T_dev[labels][:, labels]
        disc_compat  = (disc_w > self.disc_compat_thresh).float()
        pos_pair_w   = pos_mask * (1.0 + (self.disc_pos_scale - 1.0) * disc_compat)

        disc_distant = (disc_w <= self.disc_compat_thresh).float()
        neg_mask     = (pos_mask + self_mask < 1.0).float()
        neg_pair_w   = 1.0 + (self.disc_neg_scale - 1.0) * disc_distant * neg_mask

        n_pos_per_anchor = pos_mask.sum(dim=1)
        valid = n_pos_per_anchor >= SCL_MIN_POSITIVES
        if not valid.any():
            return torch.tensor(0.0, device=device, requires_grad=True)

        logits_mask  = 1.0 - self_mask
        weighted_sim = sim * neg_pair_w * logits_mask + sim * self_mask
        exp_sim      = torch.exp(
            weighted_sim - weighted_sim.max(dim=1, keepdim=True).values
        ) * logits_mask
        log_denom    = torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)
        log_prob     = sim - log_denom

        weighted_pos     = pos_pair_w * log_prob
        n_weighted_pos   = pos_pair_w.sum(dim=1).clamp(min=1e-6)
        mean_log_prob_pos = weighted_pos.sum(dim=1) / n_weighted_pos

        loss_per_anchor = (
            -(self.temperature / self.base_temperature)
            * mean_log_prob_pos * anchor_w
        )
        return loss_per_anchor[valid].mean()


# ═══════════════════════════════════════════════════════════
# BALANCED CONTRASTIVE SAMPLER
# ═══════════════════════════════════════════════════════════
class BalancedContrastiveSampler:
    def __init__(self, samples_per_class=SCL_SAMPLES_PER_CLASS,
                 noise_std=0.01, rare_ids=None):
        self.samples_per_class = samples_per_class
        self.noise_std         = noise_std
        self.rare_ids          = set(rare_ids or [])

    def sample(self, embeddings: torch.Tensor,
               labels: torch.Tensor) -> tuple:
        device = embeddings.device
        emb_list, lbl_list = [], []
        for lbl in labels.unique().tolist():
            idx  = (labels == lbl).nonzero(as_tuple=True)[0]
            embs = embeddings[idx]
            k    = embs.shape[0]
            target = self.samples_per_class
            if k >= target:
                chosen = torch.randperm(k, device=device)[:target]
                emb_list.append(embs[chosen])
                lbl_list.append(torch.full((target,), lbl,
                                           dtype=torch.long, device=device))
            else:
                emb_list.append(embs)
                n_needed = target - k
                if int(lbl) in self.rare_ids or k < target // 2:
                    src   = embs[torch.randint(0, k, (n_needed,), device=device)]
                    noise = torch.randn_like(src) * self.noise_std
                    emb_list.append(src + noise)
                    lbl_list.append(torch.full((k + n_needed,), lbl,
                                               dtype=torch.long, device=device))
                else:
                    lbl_list.append(torch.full((k,), lbl,
                                               dtype=torch.long, device=device))
        if not emb_list:
            return embeddings, labels
        out_embs = torch.cat(emb_list, dim=0)
        out_labs = torch.cat(lbl_list, dim=0)
        perm = torch.randperm(out_embs.shape[0], device=device)
        return out_embs[perm], out_labs[perm]


# ═══════════════════════════════════════════════════════════
# RARE-ONLY INTRA-CLASS MANIFOLD MIXUP
# ═══════════════════════════════════════════════════════════
class RareOnlyMixup(nn.Module):
    def __init__(self, alpha=MIXUP_ALPHA, noise_std=MIXUP_NOISE_STD):
        super().__init__()
        self.alpha     = alpha
        self.noise_std = noise_std

    @staticmethod
    def _beta_gpu(alpha, size, device):
        if alpha <= 0:
            return torch.ones(size, device=device)
        dist = torch.distributions.Beta(
            torch.tensor(alpha, device=device),
            torch.tensor(alpha, device=device))
        lam = dist.sample((size,))
        return torch.max(lam, 1.0 - lam)

    def forward(self, sent_vecs, labels, rare_ids_t, lengths):
        device = sent_vecs.device
        B, T, D = sent_vecs.shape
        rare_embs_by_class = defaultdict(list)
        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                l = int(labels[b, t].item())
                if l < 0:
                    continue
                if bool((rare_ids_t == l).any().item()):
                    rare_embs_by_class[l].append(sent_vecs[b, t])
        total_rare = sum(len(v) for v in rare_embs_by_class.values())
        if total_rare < MIXUP_MIN_RARE_IN_BATCH:
            return None, None, False
        mixed_list, label_list = [], []
        for cls_id, emb_list in rare_embs_by_class.items():
            embs = torch.stack(emb_list)
            K = embs.shape[0]
            if K == 1:
                noise = torch.randn_like(embs[0]) * self.noise_std
                mixed_list.append((embs[0] + noise).unsqueeze(0))
                label_list.append(torch.tensor([cls_id], dtype=torch.long, device=device))
            else:
                perm = torch.randperm(K, device=device)
                for i in range(K):
                    if perm[i] == i:
                        perm[i] = (i + 1) % K
                lam   = self._beta_gpu(self.alpha, K, device).unsqueeze(1)
                mixed = lam * embs + (1.0 - lam) * embs[perm]
                mixed_list.append(mixed)
                label_list.append(torch.full((K,), cls_id,
                                             dtype=torch.long, device=device))
        mixed_embs  = torch.cat(mixed_list, dim=0)
        hard_labels = torch.cat(label_list, dim=0)
        return mixed_embs, hard_labels, True


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.query     = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn = self.attn_drop(F.softmax(attn, dim=-1))
        return self.out_proj(torch.matmul(attn, V).squeeze(2).reshape(N, H))


# ═══════════════════════════════════════════════════════════
# BASE MODEL  (with DA-IA-SCL projection head + Rare Mixup)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    """
    sent_out_dim = 256  → KG embedding dim / SCL input dim
    ctx_out_dim  = 128  → actor/critic head input dim

    SCL projection head (scl_proj) is SEPARATE from the CRF classifier.
    It is trained in Phase A+B, then FROZEN before PPO starts.
    This ensures PPO gradients never corrupt the metric-learning geometry.
    """
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        scl_proj_dim     = SCL_PROJ_DIM,
        scl_loss_fn      = None,
        scl_class_weights: torch.Tensor = None,
        scl_sampler      = None,
        rare_ids: list   = None,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True, batch_first=True,
            dropout=dropout if sent_lstm_layers > 1 else 0.0)
        self.sent_out_dim    = sent_lstm_hidden * 2   # 256
        self.mha_pooling     = MultiHeadAttentionPooling(
            self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True, batch_first=True,
            dropout=dropout if ctx_lstm_layers > 1 else 0.0)
        self.ctx_out_dim = ctx_lstm_hidden * 2   # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels))
        self.crf     = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # ── SCL projection head (FROZEN during PPO) ──────
        self.scl_proj = nn.Sequential(
            nn.Linear(self.sent_out_dim, self.sent_out_dim),
            nn.GELU(),
            nn.Linear(self.sent_out_dim, scl_proj_dim),
        )
        # Store SCL components — used only when self.training AND scl_enabled
        self._scl_loss_fn       = scl_loss_fn
        self._scl_class_weights = scl_class_weights
        self._scl_sampler       = scl_sampler
        self._rare_ids_list     = rare_ids or []
        self._scl_enabled       = (scl_loss_fn is not None and
                                   scl_class_weights is not None)

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        enc = self.bert.encoder.layer
        for i in range(min(n_freeze, len(enc))):
            for param in enc[i].parameters():
                param.requires_grad = False
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{len(enc)-1} + pooler.\n")

    def freeze_scl_head(self):
        """Call before PPO training to prevent contrastive geometry corruption."""
        for param in self.scl_proj.parameters():
            param.requires_grad = False
        print("  🔒 SCL projection head FROZEN — PPO gradients isolated.")

    def unfreeze_scl_head(self):
        for param in self.scl_proj.parameters():
            param.requires_grad = True

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids=flat_ids[valid],
                attention_mask=flat_mask[valid],
                token_type_ids=flat_types[valid])
            token_embs[valid] = out.last_hidden_state.to(token_embs.dtype)
        token_embs = self.dropout(token_embs)
        lstm_out, _ = self.sent_bilstm(token_embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)   # (B, T, 256)

    def get_scl_embeddings(self, sent_vecs, labels):
        B, T, D = sent_vecs.shape
        flat_vecs   = sent_vecs.reshape(B * T, D)
        flat_labels = labels.reshape(B * T)
        valid       = flat_labels != -100
        flat_vecs   = flat_vecs[valid]
        flat_labels = flat_labels[valid]
        if flat_vecs.shape[0] == 0:
            return None, None
        proj = F.normalize(self.scl_proj(flat_vecs), dim=-1)
        return proj, flat_labels

    def get_emissions(self, input_ids, attention_mask, token_type_ids,
                      lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        sent_vecs_drop = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions   # (B,T,256), (B,T,128), (B,T,C)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2],
                              dtype=torch.bool, device=emissions.device)

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss   = self.ce_loss(
                emissions.reshape(B2*T2, C), labels.reshape(B2*T2))
            total_loss = crf_loss + AUX_CE_WEIGHT * ce_loss

            # DA-IA-SCL (training only; not executed during PPO)
            if self.training and self._scl_enabled:
                proj_embs, flat_labels = self.get_scl_embeddings(sent_vecs, labels)
                if proj_embs is not None and proj_embs.shape[0] >= 2:
                    if self._scl_sampler is not None:
                        proj_embs, flat_labels = self._scl_sampler.sample(
                            proj_embs, flat_labels)
                    cw = self._scl_class_weights.to(emissions.device)
                    scl_loss = self._scl_loss_fn(
                        proj_embs, flat_labels, cw,
                        rare_ids=self._rare_ids_list)
                    total_loss = total_loss + SCL_WEIGHT * scl_loss

            return total_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# CONFUSION ANALYSIS & ADAPTIVE WEIGHTS
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def compute_confusion_pairs_and_matrix(model, dataset, rare_ids,
                                        device=DEVICE, top_k=CONF_TOP_K):
    model.eval()
    loader   = DataLoader(dataset, batch_size=2, shuffle=False,
                          collate_fn=collate_rrc)
    rare_set = set(rare_ids)
    all_preds, all_trues = [], []
    for ids, attn, ttype, labels, lengths in loader:
        ids = ids.to(device); attn = attn.to(device)
        ttype = ttype.to(device); lengths = lengths.to(device)
        decoded, _ = model(ids, attn, ttype, labels=None, lengths=lengths)
        for i, seq in enumerate(decoded):
            n = int(lengths[i].item())
            all_preds.extend(seq)
            all_trues.extend(labels[i, :n].tolist())
    str_t  = [id2label[x] for x in all_trues]
    str_p  = [id2label[x] for x in all_preds]
    raw_cm = confusion_matrix(str_t, str_p, labels=LABELS)
    confusion = defaultdict(Counter)
    for true, pred in zip(all_trues, all_preds):
        if true in rare_set and pred != true:
            confusion[true][pred] += 1
    confusion_pairs = {}
    print("\n★ Confusion pair analysis (base model):")
    for rid in rare_ids:
        counter      = confusion.get(rid, Counter())
        top          = [cls for cls, _ in counter.most_common(top_k)]
        confusion_pairs[rid] = top
        names        = [(id2label[c], counter[c]) for c in top]
        name_str     = ", ".join(f"{n}({cnt})" for n, cnt in names)
        total_errors = sum(counter.values())
        print(f"  {id2label[rid]:<22} → {name_str}  (errors: {total_errors})")
    return confusion_pairs, raw_cm


def compute_adaptive_confusion_weights(raw_cm, rare_ids,
                                        base_alpha=CONF_BASE_ALPHA):
    W = torch.full((NUM_LABELS, NUM_LABELS),
                   CONF_EDGE_WEIGHT_DEFAULT, dtype=torch.float32)
    print("\n★ Adaptive confusion edge weights:")
    for i in rare_ids:
        row = raw_cm[i].copy().astype(float); row[i] = 0.0
        row_total = row.sum()
        if row_total < 1.0:
            W[i, :] = base_alpha; W[i, i] = 0.0; continue
        row_norm = row / row_total
        top3_j   = sorted(range(NUM_LABELS), key=lambda j: -row_norm[j])[:3]
        for j in range(NUM_LABELS):
            W[i, j] = 0.0 if j == i else float(
                base_alpha + (1.0 - base_alpha) * row_norm[j])
        top3_str = ", ".join(
            f"{id2label[j]}={float(W[i,j]):.3f}" for j in top3_j if j != i)
        print(f"  {id2label[i]:<20}  top-3: {top3_str}")
    return W


def save_adaptive_weight_heatmap(confusion_weights, rare_ids, out_dir=OUT_DIR):
    rare_labels = [id2label[r] for r in rare_ids]
    w_np = confusion_weights.numpy()
    sub  = w_np[np.ix_(rare_ids, list(range(NUM_LABELS)))]
    fig, ax = plt.subplots(figsize=(14, max(4, len(rare_ids))))
    sns.heatmap(sub, annot=True, fmt=".3f",
                xticklabels=LABELS, yticklabels=rare_labels,
                cmap="YlOrRd", vmin=0.0, vmax=1.0, ax=ax)
    ax.set_title("Adaptive Confusion Edge Weights  W[rare → majority]")
    plt.tight_layout()
    path = os.path.join(out_dir, "adaptive_confusion_weights.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved heatmap → {path}")


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH  (with rare-node amplification)
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    def __init__(self, emb_dim=KG_FUSION_DIM, name="global"):
        self.emb_dim       = emb_dim
        self.name          = name
        self.nodes         = defaultdict(list)
        self.intra_edges   = defaultdict(list)
        self.cross_edges   = []
        self.conf_cx_edges = []
        self._stacked      = {}

    def add_nodes(self, embeddings, label_ids, texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}

    def add_nodes_amplified(self, embeddings, label_ids, rare_ids,
                             amp_factor=RARE_AMP_FACTOR, noise_std=RARE_AMP_NOISE,
                             texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
            if int(lid) in rare_ids:
                for _ in range(amp_factor - 1):
                    noise = torch.randn_like(emb) * noise_std
                    self.nodes[lid].append({"emb": emb + noise, "text": text})
        self._stacked = {}

    def build_edges(self, intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        print(f"  Building {self.name} KG RST edges ...")
        self._stacked    = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []
        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs      = torch.stack([n["emb"] for n in node_list])
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat   = torch.mm(embs_norm, embs_norm.T)
            for i in range(N - 1):
                self.intra_edges[lid].append(
                    (i, i+1, max(0.0, float(sim_mat[i, i+1].item()))))
            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    self.intra_edges[lid].append((i, j, float(sims[j].item())))
                    sims[j] = -1; count += 1
            self._stacked[lid] = embs
        label_ids   = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                ea = self._get_stacked(la); eb = self._get_stacked(lb)
                if ea is None or eb is None:
                    continue
                sim_mat = torch.mm(F.normalize(ea, dim=-1),
                                   F.normalize(eb, dim=-1).T)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    self.cross_edges.append(
                        (la, ni, lb, nj, float(sim_mat[ni, nj].item())))
                    cross_count += 1
        n_intra = sum(len(v) for v in self.intra_edges.values())
        n_nodes = sum(len(v) for v in self.nodes.values())
        print(f"  [{self.name}] {n_nodes} nodes | "
              f"{n_intra} intra | {len(self.cross_edges)} cross")

    def add_confusion_edges(self, confusion_pairs, confusion_weights,
                             sim_floor=CONF_SIM_FLOOR,
                             max_pairs=CONF_MAX_PAIRS_PER_CLASS):
        self.conf_cx_edges = []
        total_added = 0
        print(f"  [{self.name}] Building confusion cross-edges ...")
        for rare_lid, confused_lids in confusion_pairs.items():
            embs_rare = self._get_stacked(rare_lid)
            if embs_rare is None:
                continue
            for clid in confused_lids:
                embs_conf = self._get_stacked(clid)
                if embs_conf is None:
                    continue
                edge_w  = float(confusion_weights[rare_lid, clid].item())
                nr      = F.normalize(embs_rare, dim=-1)
                nc      = F.normalize(embs_conf, dim=-1)
                sim_mat = torch.mm(nr, nc.T)
                rows, cols = torch.where(sim_mat >= sim_floor)
                if rows.shape[0] == 0:
                    best = sim_mat.argmax()
                    ri   = int(best // sim_mat.shape[1])
                    ci   = int(best %  sim_mat.shape[1])
                    self.conf_cx_edges.append((rare_lid, ri, clid, ci, edge_w))
                    total_added += 1
                    continue
                sims_flat = sim_mat[rows, cols]
                order     = sims_flat.argsort(descending=True)
                rows      = rows[order[:max_pairs]]
                cols      = cols[order[:max_pairs]]
                for ri, ci in zip(rows.tolist(), cols.tolist()):
                    self.conf_cx_edges.append((rare_lid, ri, clid, ci, edge_w))
                total_added += len(rows)
        print(f"  [{self.name}] Confusion edges added: {total_added}")
        return total_added

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        data = {
            "name": self.name,
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges":   {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges":    self.cross_edges,
            "conf_cx_edges":  self.conf_cx_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  [{self.name}] KG saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        with open(path) as f:
            data = json.load(f)
        kg = cls(emb_dim=emb_dim, name=data.get("name", "global"))
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]),
                                      "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges   = [tuple(e) for e in data.get("cross_edges",   [])]
        kg.conf_cx_edges = [tuple(e) for e in data.get("conf_cx_edges", [])]
        print(f"  [{kg.name}] KG loaded ← {path}  "
              f"(conf_cx_edges: {len(kg.conf_cx_edges)})")
        return kg


# ═══════════════════════════════════════════════════════════
# DUAL KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class DualKnowledgeGraph:
    """
    G_all — global KG, all sentences (no amplification)
    G_min — minority KG, rare sentences amplified RARE_AMP_FACTOR×
    """
    def __init__(self, rare_ids, emb_dim=KG_FUSION_DIM):
        self.rare_ids = set(rare_ids)
        self.g_all    = KnowledgeGraph(emb_dim=emb_dim, name="global")
        self.g_min    = KnowledgeGraph(emb_dim=emb_dim, name="minority")

    def add_nodes(self, embeddings, label_ids, texts=None):
        self.g_all.add_nodes(embeddings, label_ids, texts)
        min_embs, min_ids, min_texts = [], [], []
        for k, (emb, lid) in enumerate(zip(embeddings, label_ids)):
            if lid in self.rare_ids:
                min_embs.append(emb)
                min_ids.append(lid)
                if texts:
                    min_texts.append(texts[k])
        if min_embs:
            stacked = torch.stack(min_embs)
            self.g_min.add_nodes_amplified(
                stacked, min_ids, rare_ids=self.rare_ids,
                amp_factor=RARE_AMP_FACTOR, noise_std=RARE_AMP_NOISE,
                texts=min_texts if texts else None)

    def build_edges(self):
        self.g_all.build_edges()
        self.g_min.build_edges(
            intra_thresh=RST_INTRA_THRESH - 0.1,
            cross_thresh=RST_CROSS_THRESH - 0.1)

    def build_confusion_edges(self, confusion_pairs, confusion_weights):
        print("\n★ Inserting adaptive confusion edges into G_all ...")
        n_all = self.g_all.add_confusion_edges(confusion_pairs, confusion_weights)
        print("\n★ Inserting adaptive confusion edges into G_min ...")
        n_min = self.g_min.add_confusion_edges(
            confusion_pairs, confusion_weights,
            sim_floor=CONF_SIM_FLOOR - 0.05,
            max_pairs=CONF_MAX_PAIRS_PER_CLASS)
        print(f"\n  Total confusion edges: G_all={n_all} | G_min={n_min}")

    def save(self, dir_path):
        self.g_all.save(os.path.join(dir_path, "kg_global.json"))
        self.g_min.save(os.path.join(dir_path, "kg_minority.json"))

    @classmethod
    def load(cls, dir_path, rare_ids, emb_dim=KG_FUSION_DIM):
        dual = cls(rare_ids=rare_ids, emb_dim=emb_dim)
        dual.g_all = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_global.json"), emb_dim=emb_dim)
        dual.g_min = KnowledgeGraph.load(
            os.path.join(dir_path, "kg_minority.json"), emb_dim=emb_dim)
        return dual


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# MINORITY-AWARE RETRIEVER
# ═══════════════════════════════════════════════════════════
class MinorityAwareRetriever:
    def __init__(self, kg, label_freqs, rare_ids,
                 top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
                 top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA):
        self.kg           = kg
        self.rare_ids     = set(rare_ids)
        self.top_k        = top_k
        self.top_nodes_k1 = top_nodes_k1
        self.top_nodes_k2 = top_nodes_k2
        self.hop          = hop
        self.beta         = beta
        self.rarity       = {}
        for i in range(NUM_LABELS):
            freq = label_freqs.get(id2label[i], 0.0)
            self.rarity[i] = 1.0 / (math.log(freq + 1.0 + 1e-6))
        max_rar = max(self.rarity.values()) or 1.0
        self.rarity = {k: v / max_rar for k, v in self.rarity.items()}
        self._conf_by_rare = defaultdict(list)
        for edge in self.kg.conf_cx_edges:
            self._conf_by_rare[edge[0]].append(edge)

    def retrieve(self, h_i: torch.Tensor) -> list:
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm    = F.normalize(embs, dim=-1)
            base_sims    = torch.mv(embs_norm, h_norm.squeeze(0))
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            subgraph_scores[lid] = float((base_sims + rarity_bonus).max().item())
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]
        results = []
        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm    = F.normalize(embs, dim=-1)
            base_sims    = torch.mv(embs_norm, h_norm.squeeze(0))
            rarity_bonus = self.beta * self.rarity.get(lid, 0.0)
            boosted      = base_sims + rarity_bonus
            k1     = min(self.top_nodes_k1, embs.shape[0])
            k1_idx = boosted.topk(k1).indices.tolist()
            k1_sim = base_sims[k1_idx].tolist()
            seed_set = set(k1_idx)
            if lid in self.rare_ids:
                k2     = min(self.top_nodes_k2, embs.shape[0])
                k2_idx = base_sims.topk(k2).indices.tolist()
                for idx in k2_idx:
                    if idx not in seed_set:
                        seed_set.add(idx)
                        k1_sim.append(float(base_sims[idx].item()))
                        k1_idx.append(idx)
            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        ce = self.kg._get_stacked(lb)
                        if ce is not None and nj < ce.shape[0]:
                            results.append((ce[nj], w))
                    elif lb == lid and nj in seed_set:
                        ce = self.kg._get_stacked(la)
                        if ce is not None and ni < ce.shape[0]:
                            results.append((ce[ni], w))
            for idx, sim in zip(k1_idx, k1_sim):
                results.append((embs[idx], float(sim)))
        for lid in selected:
            if lid not in self.rare_ids:
                continue
            for edge in self._conf_by_rare.get(lid, []):
                _, _, clid, cnidx, edge_w = edge
                c_embs = self.kg._get_stacked(clid)
                if c_embs is None or cnidx >= c_embs.shape[0]:
                    continue
                results.append((c_embs[cnidx], edge_w))
        return results


# ═══════════════════════════════════════════════════════════
# DUAL KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class DualKGRetriever:
    def __init__(self, dual_kg, label_freqs, rare_ids):
        self.retriever_all = MinorityAwareRetriever(
            kg=dual_kg.g_all, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA)
        self.retriever_min = MinorityAwareRetriever(
            kg=dual_kg.g_min, label_freqs=label_freqs, rare_ids=rare_ids,
            top_k=KG_TOP_K, top_nodes_k1=KG_TOP_NODES,
            top_nodes_k2=KG_MIN_NODES, hop=KG_HOP, beta=RARITY_BETA * 1.5)

    def retrieve(self, h_i):
        return self.retriever_all.retrieve(h_i), self.retriever_min.retrieve(h_i)


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION + DYNAMIC FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i, neighbours, device=None):
        if not neighbours:
            return torch.zeros_like(h_i)
        if device is None:
            device = h_i.device
        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)
        q     = self.proj_q(h_i.unsqueeze(0))
        k     = self.proj_k(embs)
        dot   = torch.mv(k, q.squeeze(0)) * self.scale
        alpha = F.softmax(dot * weights, dim=0)
        alpha = self.dropout(alpha)
        return (alpha.unsqueeze(-1) * embs).sum(dim=0)


class DynamicFusion(nn.Module):
    def __init__(self, emb_dim=KG_FUSION_DIM):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(emb_dim * 3, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, h_i, v_all, v_min, entropy):
        alpha_fixed = (ALPHA_HIGH_ENTROPY if entropy > UNCERTAINTY_THRESH
                       else ALPHA_LOW_ENTROPY)
        gate_val = self.gate(
            torch.cat([h_i, v_all, v_min], dim=-1).unsqueeze(0)).squeeze()
        alpha = 0.5 * alpha_fixed + 0.5 * gate_val.item()
        return max(0.2, min(0.8, alpha)) * v_all + \
               (1.0 - max(0.2, min(0.8, alpha))) * v_min


# ═══════════════════════════════════════════════════════════
# DUAL KG AUGMENTED MODEL  (Phase B)
#   + DA-IA-SCL on fused embeddings
#   + Rare-only Mixup
# ═══════════════════════════════════════════════════════════
class DualKGAugmentedModel(nn.Module):
    """
    The SCL projection head in this model is ALSO a separate branch
    trained on fused_sent_vecs.  It is FROZEN at the start of Phase C+D
    together with base.scl_proj so that PPO never disturbs contrastive clusters.
    """
    def __init__(self, base_model, dual_kg, dual_retriever, rare_ids,
                 scl_loss_fn=None, scl_class_weights=None, scl_sampler=None,
                 scl_proj_dim=SCL_PROJ_DIM):
        super().__init__()
        self.base        = base_model
        self.dual_kg     = dual_kg
        self.retriever   = dual_retriever
        self.rare_ids    = set(rare_ids)
        self.rare_list   = sorted(rare_ids)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim   # 256
        ctx_dim  = base_model.ctx_out_dim    # 128

        self.gat_all     = GraphAttentionFusion(emb_dim=sent_dim)
        self.gat_min     = GraphAttentionFusion(emb_dim=sent_dim)
        self.dyn_fusion  = DynamicFusion(emb_dim=sent_dim)
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(), nn.Dropout(DROPOUT))
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2), nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS))
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # SCL projection on fused embeddings (FROZEN during PPO)
        self.fused_scl_proj = nn.Sequential(
            nn.Linear(sent_dim, sent_dim), nn.GELU(),
            nn.Linear(sent_dim, scl_proj_dim))
        self._scl_loss_fn       = scl_loss_fn
        self._scl_class_weights = scl_class_weights
        self._scl_sampler       = scl_sampler
        self._scl_enabled       = (scl_loss_fn is not None and
                                   scl_class_weights is not None)

        # Rare-only Mixup
        self.rare_mixup = RareOnlyMixup(alpha=MIXUP_ALPHA, noise_std=MIXUP_NOISE_STD)
        self.mixup_cls  = nn.Sequential(
            nn.Linear(sent_dim, ctx_dim), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, NUM_LABELS))
        self.mixup_ce   = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        self.register_buffer("_rare_t",
                             torch.tensor(self.rare_list, dtype=torch.long))

    def freeze_scl_heads(self):
        """Freeze all SCL projection heads before PPO."""
        self.base.freeze_scl_head()
        for param in self.fused_scl_proj.parameters():
            param.requires_grad = False
        print("  🔒 Fused SCL projection head FROZEN.")

    def _get_fused_scl_embs(self, fused_sent, labels):
        B, T, D = fused_sent.shape
        flat_vecs   = fused_sent.reshape(B * T, D)
        flat_labels = labels.reshape(B * T)
        valid       = flat_labels != -100
        flat_vecs   = flat_vecs[valid]
        flat_labels = flat_labels[valid]
        if flat_vecs.shape[0] == 0:
            return None, None
        proj = F.normalize(self.fused_scl_proj(flat_vecs), dim=-1)
        return proj, flat_labels

    def _fuse_batch(self, sent_vecs, emissions, lengths, device):
        B, T, sent_dim = sent_vecs.shape
        fused          = sent_vecs.clone()
        entropy_map    = self.uncertainty.entropy(emissions)
        uncertain_mask = entropy_map > UNCERTAINTY_THRESH
        top_labels     = self.uncertainty.top_label(emissions)
        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids
                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue
                h_i_cpu  = sent_vecs[b, t].detach().cpu()
                h_all, h_min = self.retriever.retrieve(h_i_cpu)
                if not h_all and not h_min:
                    continue
                h_i_gpu = sent_vecs[b, t]
                ent_val  = float(entropy_map[b, t].item())
                v_all = (self.gat_all(h_i_gpu, h_all, device=device)
                         if h_all else torch.zeros(sent_dim, device=device))
                v_min = (self.gat_min(h_i_gpu, h_min, device=device)
                         if h_min else torch.zeros(sent_dim, device=device))
                v_i = self.dyn_fusion(h_i_gpu, v_all, v_min, ent_val)
                fused[b, t] = h_i_gpu + v_i
        return fused

    def get_fused_ctx(self, input_ids, attention_mask, token_type_ids, lengths=None):
        """
        Expose all intermediate tensors for PPOModel.
        Returns: fused_sent(B,T,256), fused_ctx(B,T,128),
                 base_emissions(B,T,C), fused_emissions(B,T,C)
        """
        device = input_ids.device
        sent_vecs, _, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)
        fused_sent = self._fuse_batch(sent_vecs, base_emissions, lengths, device)
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)
        fused_emissions = torch.nan_to_num(
            self.fusion_classifier(fused_ctx), nan=0.0, posinf=1e4, neginf=-1e4)
        return fused_sent, fused_ctx, base_emissions, fused_emissions

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        fused_sent, fused_ctx, base_emissions, fused_emissions = (
            self.get_fused_ctx(input_ids, attention_mask,
                               token_type_ids, lengths=lengths))
        device = input_ids.device
        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2],
                              dtype=torch.bool, device=device)

        combined = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf_loss  = -self.base.crf(
                base_emissions, safe, mask=mask, reduction="mean")
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = combined.shape
            ce_loss = self.ce_loss(
                combined.reshape(B2*T2, C), labels.reshape(B2*T2))
            total_loss = ((base_crf_loss + fused_crf_loss) / 2.0
                          + AUX_CE_WEIGHT * ce_loss)

            # DA-IA-SCL on fused embeddings (training only; NOT in PPO)
            if self.training and self._scl_enabled:
                proj_f, flat_lf = self._get_fused_scl_embs(fused_sent, labels)
                if proj_f is not None and proj_f.shape[0] >= 2:
                    if self._scl_sampler is not None:
                        proj_f, flat_lf = self._scl_sampler.sample(proj_f, flat_lf)
                    cw = self._scl_class_weights.to(device)
                    scl_loss = self._scl_loss_fn(
                        proj_f, flat_lf, cw, rare_ids=self.rare_list)
                    total_loss = total_loss + SCL_WEIGHT * scl_loss

            # Rare-only Mixup (training only; NOT in PPO)
            if self.training:
                mx_embs, mx_labs, valid = self.rare_mixup(
                    fused_sent, labels, self._rare_t, lengths)
                if valid and mx_embs is not None:
                    mx_logits  = self.mixup_cls(mx_embs)
                    mx_loss    = self.mixup_ce(mx_logits, mx_labs)
                    total_loss = total_loss + MIXUP_LOSS_WEIGHT * mx_loss

            return total_loss, combined
        else:
            return self.fusion_crf.decode(fused_emissions, mask=mask), fused_emissions


# ═══════════════════════════════════════════════════════════
# PPO COMPONENTS
# ═══════════════════════════════════════════════════════════
class ActorHead(nn.Module):
    def __init__(self, ctx_dim, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(ctx_dim // 2, num_labels))

    def forward(self, ctx_out):
        return self.net(ctx_out)

    def copy_from_fusion_classifier(self, fusion_classifier):
        try:
            self.net.load_state_dict(fusion_classifier.state_dict())
            print("  ✔ ActorHead warm-started from Phase B fusion_classifier.")
        except Exception as e:
            print(f"  ⚠ ActorHead warm-start failed ({e}), using random init.")


class CriticHead(nn.Module):
    def __init__(self, ctx_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Linear(ctx_dim // 2, 1))

    def forward(self, ctx_out):
        return self.net(ctx_out).squeeze(-1)


class PPOModel(nn.Module):
    """
    PPO wrapper around DualKGAugmentedModel.

    CRITICAL isolation guarantee:
      • Before __init__ is called, kg_model.freeze_scl_heads() MUST be called.
      • ActorHead and CriticHead are NEW parameters — only they receive PPO grads.
      • kg_model (base encoder + KG fusion) receives PPO grads at lr × 0.1
        for the CRF/classifier heads only; the scl_proj is frozen.
      • SCL and Mixup losses are gated by self.training in DualKGAugmentedModel,
        and DualKGAugmentedModel.forward() is NOT called during PPO rollout/update.
        PPO calls get_fused_ctx() directly, bypassing SCL/Mixup branches.
    """
    def __init__(self, kg_model: DualKGAugmentedModel, rare_ids: list):
        super().__init__()
        self.kg_model = kg_model
        self.rare_ids = set(rare_ids)
        ctx_dim       = kg_model.base.ctx_out_dim   # 128
        self.actor    = ActorHead(ctx_dim)
        self.critic   = CriticHead(ctx_dim)
        self.actor.copy_from_fusion_classifier(kg_model.fusion_classifier)

    def get_ctx_out(self, input_ids, attention_mask, token_type_ids, lengths):
        """Returns: fused_sent(B,T,256), fused_ctx(B,T,128), base_emissions(B,T,C)"""
        fused_sent, fused_ctx, base_emissions, _ = (
            self.kg_model.get_fused_ctx(
                input_ids, attention_mask, token_type_ids, lengths=lengths))
        return fused_sent, fused_ctx, base_emissions

    @torch.no_grad()
    def decode_actions(self, actor_logits, lengths):
        B, T, _ = actor_logits.shape
        mask = torch.zeros(B, T, dtype=torch.bool, device=actor_logits.device)
        for i, l in enumerate(lengths):
            mask[i, :l] = True
        return self.kg_model.fusion_crf.decode(actor_logits, mask=mask)


class RewardShaper:
    def __init__(self, rare_ids):
        self.rare_ids = set(rare_ids)

    def base_reward(self, pred, true):
        correct = (pred == true)
        if true in self.rare_ids:
            return REWARD_RARE_CORRECT if correct else REWARD_RARE_WRONG
        return REWARD_MAJ_CORRECT if correct else REWARD_MAJ_WRONG


def compute_gae(rewards, values, gamma=GAE_GAMMA, lam=GAE_LAMBDA):
    T          = len(rewards)
    advantages = [0.0] * T
    returns    = [0.0] * T
    gae        = 0.0
    for t in reversed(range(T)):
        nv    = values[t+1] if t+1 < T else 0.0
        delta = rewards[t] + gamma * nv - values[t]
        gae   = delta + gamma * lam * gae
        advantages[t] = gae
        returns[t]    = gae + values[t]
    return advantages, returns


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states        = []
        self.actor_logits  = []
        self.actions       = []
        self.true_labels   = []
        self.advantages    = []
        self.returns       = []
        self.old_log_probs = []

    def add_trajectory(self, states_t, logits_t, actions_t,
                       trues_t, advantages_t, returns_t):
        for i in range(len(actions_t)):
            self.states.append(states_t[i])
            self.actor_logits.append(logits_t[i])
            self.actions.append(int(actions_t[i]))
            self.true_labels.append(int(trues_t[i]))
            self.advantages.append(float(advantages_t[i]))
            self.returns.append(float(returns_t[i]))
            lp = F.log_softmax(logits_t[i], dim=-1)[int(actions_t[i])]
            self.old_log_probs.append(float(lp.item()))

    def __len__(self):
        return len(self.actions)

    def as_tensors(self, device):
        states        = torch.stack(self.states).to(device)
        actor_logits  = torch.stack(self.actor_logits).to(device)
        actions       = torch.tensor(self.actions,       dtype=torch.long,  device=device)
        advantages    = torch.tensor(self.advantages,    dtype=torch.float, device=device)
        returns       = torch.tensor(self.returns,       dtype=torch.float, device=device)
        old_log_probs = torch.tensor(self.old_log_probs, dtype=torch.float, device=device)
        advantages    = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return states, actor_logits, actions, advantages, returns, old_log_probs


# ═══════════════════════════════════════════════════════════
# PHASE D: CURRICULUM + SELF-PLAY + KG DISAGREE
# ═══════════════════════════════════════════════════════════
class SyntheticCorrectionModel(nn.Module):
    def __init__(self, ctx_dim, num_labels=NUM_LABELS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ctx_dim + num_labels * 2, hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1), nn.Tanh())

    def forward(self, ctx_emb, pred_ids, true_ids):
        C = NUM_LABELS
        x = torch.cat([
            ctx_emb,
            F.one_hot(pred_ids, C).float(),
            F.one_hot(true_ids, C).float()], dim=-1)
        return self.net(x).squeeze(-1)


def collect_correction_data(ppo_model, train_dataset, rare_ids, device=DEVICE):
    ppo_model.eval()
    loader   = DataLoader(train_dataset, batch_size=1, shuffle=False,
                          collate_fn=collate_rrc)
    rare_set = set(rare_ids)
    data     = []
    with torch.no_grad():
        for ids, attn, ttype, labels, lengths in loader:
            ids = ids.to(device); attn = attn.to(device)
            ttype = ttype.to(device); lengths = lengths.to(device)
            _, ctx_out, _ = ppo_model.get_ctx_out(ids, attn, ttype, lengths)
            actor_logits  = ppo_model.actor(ctx_out)
            probs         = F.softmax(actor_logits, dim=-1)
            n = int(lengths[0].item())
            for t in range(n):
                true_lbl = int(labels[0, t].item())
                if true_lbl < 0 or true_lbl not in rare_set:
                    continue
                pred_lbl   = int(actor_logits[0, t].argmax().item())
                confidence = float(probs[0, t].max().item())
                ctx_emb    = ctx_out[0, t].cpu()
                if pred_lbl != true_lbl:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": -1.0})
                elif confidence >= 0.7:
                    data.append({"ctx_emb": ctx_emb, "pred": pred_lbl,
                                 "true": true_lbl, "target": +1.0})
    neg = [d for d in data if d["target"] < 0]
    pos = [d for d in data if d["target"] > 0]
    print(f"  Correction data: {len(neg)} errors, {len(pos)} confident-correct")
    return data


def train_correction_model(corr_model, correction_data, device=DEVICE):
    if not correction_data:
        print("  ⚠ No correction data.")
        return corr_model
    optimizer = torch.optim.AdamW(corr_model.parameters(),
                                   lr=CORRECTION_LR, weight_decay=0.01)
    corr_model.to(device).train()
    neg_data = [d for d in correction_data if d["target"] < 0]
    pos_data = [d for d in correction_data if d["target"] > 0]
    for epoch in range(CORRECTION_EPOCHS):
        if pos_data:
            n_pos  = min(len(pos_data), 64)
            n_neg  = min(len(neg_data), n_pos * CORRECTION_NEG_RATIO)
            batch  = random.sample(pos_data, n_pos) + random.sample(neg_data, n_neg)
        else:
            batch  = random.sample(neg_data, min(len(neg_data), 256))
        random.shuffle(batch)
        ctx_embs = torch.stack([b["ctx_emb"] for b in batch]).to(device)
        preds    = torch.tensor([b["pred"]   for b in batch], dtype=torch.long,  device=device)
        trues    = torch.tensor([b["true"]   for b in batch], dtype=torch.long,  device=device)
        targets  = torch.tensor([b["target"] for b in batch], dtype=torch.float, device=device)
        loss = F.mse_loss(corr_model(ctx_embs, preds, trues), targets)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(corr_model.parameters(), 1.0)
        optimizer.step()
        if (epoch + 1) % 5 == 0:
            print(f"    [Correction] epoch {epoch+1}/{CORRECTION_EPOCHS} "
                  f"loss={loss.item():.4f}")
    torch.save(corr_model.state_dict(),
               os.path.join(OUT_DIR, "correction_model.bin"))
    print("  ✔ Correction model saved.")
    return corr_model


def kg_disagree_corrections(dual_retriever, sent_vecs_cpu, actor_preds,
                              rare_ids, bonus=KG_DISAGREE_BONUS):
    rare_set    = set(rare_ids)
    corrections = []
    for h_i, pred in zip(sent_vecs_cpu, actor_preds):
        h_all, _ = dual_retriever.retrieve(h_i)
        if not h_all:
            corrections.append(0.0)
            continue
        total_rare_sim = sum(max(w, 0.0) for _, w in h_all)
        apply_bonus    = (pred not in rare_set) and (total_rare_sim > 0.5)
        corrections.append(bonus if apply_bonus else 0.0)
    return corrections


class CurriculumScheduler:
    def __init__(self, rare_ids, num_labels=NUM_LABELS,
                 window=CURRICULUM_WINDOW, min_w=CURRICULUM_MIN_W,
                 max_w=CURRICULUM_MAX_W):
        self.rare_ids    = set(rare_ids)
        self.num_labels  = num_labels
        self.window      = window
        self.min_w       = min_w
        self.max_w       = max_w
        self.f1_history  = {i: deque(maxlen=window) for i in range(num_labels)}
        self.multipliers = {i: max_w if i in self.rare_ids else 1.0
                            for i in range(num_labels)}

    def update(self, all_trues, all_preds):
        per_class_f1 = f1_score(all_trues, all_preds,
                                 labels=list(range(self.num_labels)),
                                 average=None, zero_division=0)
        for i in range(self.num_labels):
            self.f1_history[i].append(float(per_class_f1[i]))
        for i in self.rare_ids:
            hist = list(self.f1_history[i])
            if len(hist) < 2:
                continue
            mean_f1  = np.mean(hist)
            trend    = hist[-1] - hist[0]
            target_w = self.max_w * (1.0 - mean_f1) + self.min_w * mean_f1
            if trend > 0.01:
                target_w *= 0.9
            self.multipliers[i] = float(np.clip(
                0.7 * self.multipliers[i] + 0.3 * target_w, self.min_w, self.max_w))

    def get_multiplier(self, label_id):
        return self.multipliers.get(label_id, 1.0)

    def log(self):
        mults = {id2label[i]: f"{self.multipliers[i]:.2f}"
                 for i in sorted(self.rare_ids) if i in self.multipliers}
        print(f"  Curriculum multipliers: {mults}")


def compute_shaped_reward(pred, true, rare_ids, base_shaper,
                           correction_model=None, ctx_emb_cpu=None,
                           curriculum=None, kg_delta=0.0, device=DEVICE):
    r = base_shaper.base_reward(pred, true)
    if curriculum is not None and true in rare_ids:
        r *= curriculum.get_multiplier(true)
    r += kg_delta
    if correction_model is not None and ctx_emb_cpu is not None:
        with torch.no_grad():
            ce = ctx_emb_cpu.unsqueeze(0).to(device)
            pd = torch.tensor([pred], dtype=torch.long, device=device)
            td = torch.tensor([true], dtype=torch.long, device=device)
            r += float(correction_model(ce, pd, td).item())
    return r


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1      = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1      = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1   = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec     = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec     = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec  = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc           = accuracy_score(all_trues, all_preds)
    per_class_f1   = f1_score(all_trues, all_preds,
                               labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds,
                                     labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds,
                                  labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1":        float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)}
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0
    str_t      = [id2label[x] for x in all_trues]
    str_p      = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_t, str_p, labels=LABELS,
                                        digits=4, zero_division=0)
    cm = confusion_matrix(str_t, str_p, labels=LABELS)
    return dict(
        macro_f1=macro_f1, micro_f1=micro_f1, weighted_f1=weighted_f1,
        macro_precision=macro_prec, micro_precision=micro_prec,
        weighted_precision=weighted_prec,
        macro_recall=macro_rec, micro_recall=micro_rec,
        weighted_recall=weighted_rec,
        rare_f1=rare_f1, rare_precision=rare_prec, rare_recall=rare_rec,
        per_class_metrics=per_class_metrics,
        accuracy=acc, cls_report=cls_report, cm=cm,
        all_preds=all_preds, all_trues=all_trues)


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if     p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# BASE TRAINER  (Phase A)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        pg = []
        pg.append({"params": list(self.model.bert.pooler.parameters()),
                   "lr": BERT_LR, "weight_decay": WEIGHT_DECAY})
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth  = (n - 1) - i
            lr_i   = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                pg.append({"params": params, "lr": lr_i,
                           "weight_decay": WEIGHT_DECAY})
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf, self.model.scl_proj]
        head_params = [p for m in head_modules for p in m.parameters()]
        pg.append({"params": head_params, "lr": HEAD_LR,
                   "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti = time.time() - infer_start
            ns = len(all_trues)
            with open(os.path.join(OUT_DIR,
                                   f"inference_time_{split_name}.json"), "w") as f:
                json.dump({"total_inference_time_s": ti,
                           "latency_per_document_ms": ti/max(1,n_samples)*1000,
                           "throughput_sentences_per_s": ns/max(1e-9,ti)}, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids, tokenizer,
              num_epochs=NUM_EPOCHS_BASE):
        train_loader  = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer     = self.build_optimizer()
        total_steps   = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps  = int(WARMUP_RATIO * total_steps)
        scheduler     = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping(patience=ES_PATIENCE)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time  = time.time() - epoch_start
            avg_loss    = running_loss / max(1, n_steps)
            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base+DA-IA-SCL] Ep {epoch:03d}/{num_epochs} | "
                f"loss:{avg_loss:.4f} val:{val_loss:.4f} | "
                f"mac_F1:{val_metrics['macro_f1']:.4f} "
                f"rare_F1:{val_metrics['rare_f1']:.4f} | "
                f"t:{epoch_time:.1f}s ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "base_da_ia_scl",
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Base early stopping at epoch {epoch}.\n")
                break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "base_da_ia_scl_model.bin"))
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# DUAL KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_dual_knowledge_graph(base_model, train_docs, tokenizer, rare_ids,
                                 confusion_pairs=None, confusion_weights=None,
                                 device=DEVICE) -> DualKnowledgeGraph:
    print(f"\n🔨 Building Dual KG (G_all + G_min×{RARE_AMP_FACTOR}) ...")
    base_model.eval(); base_model.to(device)
    rare_set      = set(rare_ids)
    dual_kg       = DualKnowledgeGraph(rare_ids=rare_ids,
                                        emb_dim=base_model.sent_out_dim)
    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader        = DataLoader(dummy_dataset, batch_size=1,
                               shuffle=False, collate_fn=collate_rrc)
    n_rare_total  = 0
    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids = ids.to(device); attn = attn.to(device); ttype = ttype.to(device)
        lengths = lengths.to(device)
        sent_vecs = base_model.encode_sentences(ids, attn, ttype).squeeze(0)
        n         = int(lengths[0].item())
        dual_kg.add_nodes(sent_vecs[:n].cpu(), labels[0, :n].tolist())
        n_rare_total += sum(1 for l in labels[0, :n].tolist() if l in rare_set)
        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1}/{len(loader)} docs "
                  f"(rare sents: {n_rare_total})")
    dual_kg.build_edges()
    if confusion_pairs is not None and confusion_weights is not None:
        dual_kg.build_confusion_edges(confusion_pairs, confusion_weights)
    else:
        print("  ⚠ Skipping confusion edges.")
    dual_kg.save(OUT_DIR)
    n_all = sum(len(v) for v in dual_kg.g_all.nodes.values())
    n_min = sum(len(v) for v in dual_kg.g_min.nodes.values())
    n_rare_g_all = sum(len(dual_kg.g_all.nodes.get(r, [])) for r in rare_ids)
    n_rare_g_min = sum(len(dual_kg.g_min.nodes.get(r, [])) for r in rare_ids)
    print(f"\n  G_all: {n_all} nodes (rare: {n_rare_g_all})")
    print(f"  G_min: {n_min} nodes (rare: {n_rare_g_min}  "
          f"= {n_rare_total} orig × {RARE_AMP_FACTOR}x amp)")
    return dual_kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER  (Phase B — includes DA-IA-SCL + Rare Mixup)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: DualKGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (
            list(self.model.gat_all.parameters()) +
            list(self.model.gat_min.parameters()) +
            list(self.model.dyn_fusion.parameters()) +
            list(self.model.fusion_proj.parameters()) +
            list(self.model.fusion_classifier.parameters()) +
            list(self.model.fusion_crf.parameters()) +
            list(self.model.fused_scl_proj.parameters()) +
            list(self.model.mixup_cls.parameters()) +
            list(self.model.rare_mixup.parameters())
        )
        base_trainable = [p for p in self.model.base.parameters()
                          if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR, "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, sp in enumerate(decoded):
                    tl = int(lengths[i].item())
                    all_preds.extend(sp)
                    all_trues.extend(labels[i, :tl].tolist())
        if measure_inference_time:
            ti = time.time() - infer_start; ns = len(all_trues)
            with open(os.path.join(OUT_DIR,
                                   f"kg_inference_{split_name}.json"), "w") as f:
                json.dump({"total_inference_time_s": ti,
                           "latency_per_document_ms": ti/max(1,n_samples)*1000,
                           "throughput_sentences_per_s": ns/max(1e-9,ti)}, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader  = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer     = self.build_optimizer()
        total_steps   = len(train_loader) * num_epochs
        warmup_steps  = int(WARMUP_RATIO * total_steps)
        scheduler     = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps)
        early_stopper = EarlyStopping(patience=ES_PATIENCE_KG)
        history       = []
        best_f1, best_state = -1.0, None
        total_start   = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps = 0.0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for ids, attn, ttype, labels, lengths in train_loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            epoch_time  = time.time() - epoch_start
            avg_loss    = running_loss / max(1, n_steps)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[DualKG+DA-IA-SCL+Mixup] Ep {epoch:02d}/{num_epochs} | "
                f"loss:{avg_loss:.4f} | "
                f"mac_F1:{val_metrics['macro_f1']:.4f} "
                f"rare_F1:{val_metrics['rare_f1']:.4f} | "
                f"t:{epoch_time:.1f}s ES:{early_stopper.counter}/{early_stopper.patience}"
            )
            history.append({
                "epoch": epoch, "phase": "dual_kg_da_ia_scl_mixup",
                "train_loss": avg_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1":  val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Phase B early stopping at epoch {epoch}.\n")
                break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "dual_kg_da_ia_scl_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR,
                                    "dual_kg_da_ia_scl_mixup_model.bin"))
            print(f"\n✔ Best Phase B model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# PPO TRAINER  (Phase C+D — SCL heads already frozen)
# ═══════════════════════════════════════════════════════════
class PPOTrainer:
    """
    ISOLATION GUARANTEE:
      • kg_model.freeze_scl_heads() called before PPOModel is instantiated.
      • PPO rollout uses get_fused_ctx() → this does NOT trigger SCL or Mixup
        (those branches are inside DualKGAugmentedModel.forward() which is NOT
        called here; only get_fused_ctx() is called).
      • PPO update iterates over stored ctx_out (128-d) tensors from the buffer
        — no new BERT forward pass; SCL proj head is never in the compute graph.
      • Result: PPO reward shaping benefits from SCL-shaped sent_vecs geometry
        (through KG retrieval and KG disagreement) without any gradient coupling.
    """
    def __init__(self, ppo_model: PPOModel, rare_ids: list,
                 dual_retriever: DualKGRetriever = None, device=DEVICE):
        self.model            = ppo_model.to(device)
        self.device           = device
        self.rare_ids         = set(rare_ids)
        self.dual_retriever   = dual_retriever
        self.base_shaper      = RewardShaper(list(rare_ids))
        self.correction_model = None
        self.curriculum       = None
        self._optimizer       = None

    def build_optimizer(self):
        # Only ActorHead + CriticHead get full PPO_LR
        # KG fusion layers get PPO_LR * 0.1 (fine-tune gently)
        # SCL proj heads are frozen — NOT in this optimizer
        actor_critic_params = (list(self.model.actor.parameters()) +
                               list(self.model.critic.parameters()))
        # KG-fusion trainable params (excl. frozen scl_proj)
        kg_trainable = [
            p for name, p in self.model.kg_model.named_parameters()
            if p.requires_grad and "scl_proj" not in name
        ]
        return torch.optim.AdamW([
            {"params": actor_critic_params, "lr": PPO_LR,       "weight_decay": 0.01},
            {"params": kg_trainable,        "lr": PPO_LR * 0.1, "weight_decay": 0.01},
        ])

    @torch.no_grad()
    def collect_rollout(self, loader_iter, loader, n_docs=ROLLOUT_DOCS):
        self.model.eval()
        buffer    = RolloutBuffer()
        collected = 0

        while collected < n_docs:
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(loader)
                batch = next(loader_iter)

            ids, attn, ttype, labels, lengths = batch
            ids = ids.to(self.device); attn = attn.to(self.device)
            ttype = ttype.to(self.device); lengths = lengths.to(self.device)

            # get_fused_ctx: sent_vecs(256) for KG, ctx_out(128) for actor/critic
            # SCL and Mixup branches are NOT triggered here
            sent_vecs, ctx_out, _ = self.model.get_ctx_out(
                ids, attn, ttype, lengths)
            actor_logits = self.model.actor(ctx_out)
            values       = self.model.critic(ctx_out)
            decoded      = self.model.decode_actions(actor_logits, lengths)

            for b in range(ids.shape[0]):
                n         = int(lengths[b].item())
                actions_b = decoded[b][:n]
                trues_b   = labels[b, :n].tolist()
                if any(t < 0 for t in trues_b):
                    continue

                # KG disagreement uses sent_vecs (256-d, SCL-shaped geometry)
                if (self.dual_retriever is not None and
                        PHASE_D_MODE in ("kg_disagree", "combined")):
                    sent_cpu = [sent_vecs[b, t].detach().cpu() for t in range(n)]
                    kg_delts = kg_disagree_corrections(
                        self.dual_retriever, sent_cpu,
                        actions_b, list(self.rare_ids))
                else:
                    kg_delts = [0.0] * n

                rewards_b = []
                for t in range(n):
                    r = compute_shaped_reward(
                        pred=actions_b[t], true=trues_b[t],
                        rare_ids=self.rare_ids,
                        base_shaper=self.base_shaper,
                        correction_model=(
                            self.correction_model
                            if PHASE_D_MODE in ("self_play", "combined") else None),
                        ctx_emb_cpu=ctx_out[b, t].detach().cpu(),
                        curriculum=(
                            self.curriculum
                            if PHASE_D_MODE in ("curriculum", "combined") else None),
                        kg_delta=kg_delts[t],
                        device=self.device)
                    rewards_b.append(r)

                vals_b         = values[b, :n].detach().cpu().tolist()
                adv_b, ret_b   = compute_gae(rewards_b, vals_b)

                buffer.add_trajectory(
                    states_t    =[ctx_out[b, t].detach() for t in range(n)],
                    logits_t    =[actor_logits[b, t].detach() for t in range(n)],
                    actions_t   =actions_b,
                    trues_t     =trues_b,
                    advantages_t=adv_b,
                    returns_t   =ret_b)
                collected += 1

        return buffer, loader_iter

    def ppo_update(self, buffer: RolloutBuffer):
        self.model.train()
        (states, old_logits, actions, advantages,
         returns, old_log_probs) = buffer.as_tensors(self.device)

        N       = len(actions)
        indices = torch.randperm(N)
        total_pol, total_val, n_up = 0.0, 0.0, 0

        for _ in range(PPO_MINI_EPOCHS):
            for start in range(0, N, 256):
                idx = indices[start:start + 256]
                if len(idx) == 0:
                    continue
                s   = states[idx]; a   = actions[idx]
                adv = advantages[idx]; ret = returns[idx]
                olp = old_log_probs[idx]; oleg = old_logits[idx]

                # Forward through actor/critic heads ONLY
                # states are detached ctx_out (128-d) — no BERT/SCL in compute graph
                new_logits = self.model.actor.net(s)
                new_vals   = self.model.critic.net(s).squeeze(-1)

                new_lp  = F.log_softmax(new_logits, dim=-1).gather(
                    1, a.unsqueeze(1)).squeeze(1)
                ratio   = (new_lp - olp).exp()
                clipped = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS)

                pol_loss = -torch.min(ratio * adv, clipped * adv).mean()
                val_loss = F.mse_loss(new_vals, ret)
                entropy  = -(F.softmax(new_logits, dim=-1) *
                             F.log_softmax(new_logits, dim=-1)).sum(-1).mean()
                kl_loss  = F.kl_div(F.log_softmax(new_logits, dim=-1),
                                    F.softmax(oleg, dim=-1), reduction="batchmean")

                loss = (pol_loss
                        + PPO_VALUE_COEF    * val_loss
                        - PPO_ENTROPY_COEF  * entropy
                        + PPO_KL_COEF       * kl_loss)

                self._optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), PPO_GRAD_CLIP)
                self._optimizer.step()
                total_pol += pol_loss.item()
                total_val += val_loss.item()
                n_up      += 1

        return total_pol / max(1, n_up), total_val / max(1, n_up)

    def evaluate(self, dataset, rare_ids, split_name="dev"):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                _, ctx_out, _ = self.model.get_ctx_out(ids, attn, ttype, lengths)
                logits  = self.model.actor(ctx_out)
                decoded = self.model.decode_actions(logits, lengths)
                for i, seq in enumerate(decoded):
                    n = int(lengths[i].item())
                    all_preds.extend(seq[:n])
                    all_trues.extend(labels[i, :n].tolist())
        macro_f1 = f1_score(all_trues, all_preds, average="macro", zero_division=0)
        rare_f1  = f1_score(all_trues, all_preds,
                            labels=[r for r in rare_ids if r in all_trues],
                            average="macro", zero_division=0)
        return macro_f1, rare_f1, all_trues, all_preds

    def evaluate_full(self, dataset, rare_ids, split_name="test"):
        macro_f1, rare_f1, all_trues, all_preds = self.evaluate(
            dataset, rare_ids, split_name=split_name)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=PPO_EPOCHS):
        print(f"\n  Phase D mode : {PHASE_D_MODE}")
        print("  ══ SCL heads are FROZEN — PPO gradients isolated ══")

        if PHASE_D_MODE in ("curriculum", "combined"):
            self.curriculum = CurriculumScheduler(list(rare_ids))
            print("  ✔ Curriculum scheduler initialised.")

        if PHASE_D_MODE in ("self_play", "combined"):
            print("\n  Collecting self-play correction data ...")
            ctx_dim    = self.model.kg_model.base.ctx_out_dim   # 128
            corr_model = SyntheticCorrectionModel(ctx_dim=ctx_dim)
            corr_data  = collect_correction_data(
                self.model, train_dataset, list(rare_ids), device=self.device)
            self.correction_model = train_correction_model(
                corr_model, corr_data, device=self.device)
            self.correction_model.eval()

        self._optimizer = self.build_optimizer()
        train_loader    = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                     shuffle=True, collate_fn=collate_rrc)
        loader_iter     = iter(train_loader)
        best_rare_f1    = -1.0
        best_state      = None
        es_counter      = 0
        history         = []
        total_start     = time.time()

        print(f"\n{'='*60}")
        print("PHASE C+D: PPO RL Training")
        print(f"{'='*60}\n")

        for epoch in range(1, num_epochs + 1):
            epoch_start = time.time()
            buffer, loader_iter = self.collect_rollout(
                loader_iter, train_loader, n_docs=ROLLOUT_DOCS)
            pol_loss, val_loss  = self.ppo_update(buffer)
            macro_f1, rare_f1, all_trues, all_preds = self.evaluate(
                dev_dataset, rare_ids)

            if self.curriculum is not None:
                self.curriculum.update(all_trues, all_preds)

            epoch_time = time.time() - epoch_start
            print(
                f"[PPO] Ep {epoch:03d}/{num_epochs} | "
                f"pol:{pol_loss:.4f} val:{val_loss:.4f} | "
                f"dev_mac_F1:{macro_f1:.4f} dev_rare_F1:{rare_f1:.4f} | "
                f"buf:{len(buffer)} t:{epoch_time:.1f}s"
            )
            if self.curriculum is not None and epoch % 3 == 0:
                self.curriculum.log()

            history.append({
                "epoch": epoch, "phase": "ppo",
                "pol_loss": pol_loss, "val_loss": val_loss,
                "val_macro_f1": macro_f1, "val_rare_f1": rare_f1,
                "epoch_train_time_s": epoch_time,
            })
            if rare_f1 > best_rare_f1 + ES_MIN_DELTA:
                best_rare_f1 = rare_f1
                best_state   = {k: v.cpu().clone()
                                for k, v in self.model.state_dict().items()}
                es_counter   = 0
                print(f"  ✔ New best dev_rare_f1={best_rare_f1:.4f}")
            else:
                es_counter += 1
                if es_counter >= ES_PATIENCE_PPO:
                    print(f"\n⏹ PPO early stopping at epoch {epoch}.\n")
                    break

        total_time = time.time() - total_start
        if best_state is not None:
            self.model.load_state_dict(best_state)
            torch.save(best_state,
                       os.path.join(BEST_MODEL_DIR, "ppo_daiscl_model.bin"))
            print(f"\n✔ Best PPO model saved (dev_rare_f1={best_rare_f1:.4f})")
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "ppo_history.csv"), index=False)
        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name} Confusion Matrix  "
                 f"(DualKG + DA-IA-SCL + RareMixup + PPO)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels)
              else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name} Per-Class F1")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_discourse_matrix_heatmap(T, out_dir=OUT_DIR):
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(T.numpy(), annot=True, fmt=".2f",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="YlOrRd", vmin=0.0, vmax=1.0, ax=ax)
    ax.set_title("Discourse Transition Matrix  P(label_j | label_i)")
    ax.set_xlabel("Next label"); ax.set_ylabel("Current label")
    plt.tight_layout()
    path = os.path.join(out_dir, "discourse_transition_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_all_phases(base_df, kg_df, ppo_df):
    all_df = pd.concat([base_df, kg_df, ppo_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    b1 = len(base_df); b2 = b1 + len(kg_df)

    for df in [all_df]:
        if "train_loss" not in df.columns and "pol_loss" in df.columns:
            df["train_loss"] = df["pol_loss"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"].ffill(),
            marker="o", markersize=2, label="Train Loss")
    ax.axvline(b1, color="orange", linestyle="--", label="Phase B start")
    ax.axvline(b2, color="purple", linestyle="--", label="Phase C+D start")
    ax.set_title("Training Loss (A→B→C+D)"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"].ffill(),
            marker="o", markersize=2, label="Val Macro-F1")
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"].ffill(),
            marker="s", markersize=2, label="Val Rare-F1")
    ax.axvline(b1, color="orange", linestyle="--", label="Phase B start")
    ax.axvline(b2, color="purple", linestyle="--", label="Phase C+D start")
    ax.set_title("Val F1  (Base+SCL → DualKG+SCL+Mixup → PPO)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "all_phases_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_final_table(kg_dev, kg_test, ppo_dev, ppo_test,
                      base_time=None, kg_time=None, ppo_time=None,
                      total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    sep = "=" * 96
    print(f"\n{sep}")
    print("FINAL RESULTS — InLegalBERT + DualKG + DA-IA-SCL + RareMixup + PPO")
    print("  (SCL-PPO strictly isolated: SCL proj frozen before PPO)")
    print(sep)
    if total_trainable:
        print(f"  Trainable Parameters : {total_trainable:,}")
    if base_time:
        print(f"  Phase A time         : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B time         : {kg_time/60:.1f} min")
    if ppo_time:
        print(f"  Phase C+D time       : {ppo_time/60:.1f} min")
    print(f"  Phase D mode         : {PHASE_D_MODE}")
    print("-" * 96)
    print(f"  {'Metric':<28} {'KG-Dev':>10} {'KG-Test':>10} "
          f"{'PPO-Dev':>10} {'PPO-Test':>10}")
    print("-" * 96)
    for label, key in rows:
        print(f"  {label:<28} {kg_dev[key]:>10.4f} {kg_test[key]:>10.4f} "
              f"{ppo_dev[key]:>10.4f} {ppo_test[key]:>10.4f}")
    print(sep)
    print("\n  PER-CLASS F1 — Phase B (DualKG+SCL) vs Phase C+D (PPO) on TEST")
    print("  " + "-" * 72)
    print(f"  {'Label':<22} {'KG+SCL':>10} {'PPO':>10} {'Δ':>8}")
    print("  " + "-" * 72)
    for lbl in LABELS:
        kg_f1  = kg_test["per_class_metrics"][lbl]["f1"]
        ppo_f1 = ppo_test["per_class_metrics"][lbl]["f1"]
        delta  = ppo_f1 - kg_f1
        flag   = " ↑" if delta > 0.005 else (" ↓" if delta < -0.005 else "")
        print(f"  {lbl:<22} {kg_f1:>10.4f} {ppo_f1:>10.4f} {delta:>+8.4f}{flag}")
    print("  " + "-" * 72)


# ═══════════════════════════════════════════════════════════
# MAIN — Full pipeline  A → (A→B) → B → C+D
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device      : {DEVICE}")
    print("Architecture: InLegalBERT + BiLSTM + MHA + CRF")
    print("              → Dual KG-RAG (G_all + G_min×{} amp)".format(RARE_AMP_FACTOR))
    print("              + Adaptive Confusion Cross-Edges")
    print("              + Discourse-Aware Imbalance-Aware SCL (DA-IA-SCL)")
    print("              + Rare-only Intra-class Manifold Mixup")
    print("              + PPO RL (Phase C+D)")
    print("\n  ══ SCL-PPO isolation: SCL proj heads are FROZEN before PPO ══\n")
    print(f"  Phase D mode : {PHASE_D_MODE}\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_docs = extract_docs(load_jsonl(TRAIN_PATH))
    dev_docs   = extract_docs(load_jsonl(DEV_PATH))
    test_docs  = extract_docs(load_jsonl(TEST_PATH))
    print(f"  Train:{len(train_docs)} | Dev:{len(dev_docs)} | Test:{len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels}
                  for l in LABELS]
                 ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── Discourse transition matrix ────────────────────────
    print("\nBuilding discourse transition matrix ...")
    T_matrix = build_discourse_transition_matrix(train_docs)
    save_discourse_matrix_heatmap(T_matrix, out_dir=OUT_DIR)

    # ── SCL components ──────────────────────────────────────
    scl_class_weights = compute_scl_class_weights(label_freqs, rare_ids,
                                                   gamma=SCL_IMBAL_GAMMA)
    scl_loss_fn = DiscourseAwareSCLoss(
        temperature        = SCL_TEMPERATURE,
        minority_boost     = SCL_MINORITY_BOOST,
        disc_pos_scale     = DISC_POS_SCALE,
        disc_neg_scale     = DISC_NEG_SCALE,
        disc_compat_thresh = DISC_COMPAT_THRESH)
    scl_loss_fn.set_transition_matrix(T_matrix)
    scl_sampler = BalancedContrastiveSampler(
        samples_per_class=SCL_SAMPLES_PER_CLASS, rare_ids=rare_ids)

    print(f"\n  DA-IA-SCL: temperature={SCL_TEMPERATURE}  "
          f"minority_boost={SCL_MINORITY_BOOST}x  "
          f"disc_pos={DISC_POS_SCALE}  disc_neg={DISC_NEG_SCALE}  "
          f"imbal_gamma={SCL_IMBAL_GAMMA}  weight={SCL_WEIGHT}")

    print("Loading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ══════════════════════════════════════════════════════
    # PHASE A: InLegalBERT + BiLSTM + CRF + DA-IA-SCL
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model  (CRF + CE + DA-IA-SCL)")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        scl_loss_fn       = scl_loss_fn,
        scl_class_weights = scl_class_weights,
        scl_sampler       = scl_sampler,
        rare_ids          = rare_ids)
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE)

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Confusion Analysis + Adaptive Weights
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Confusion Analysis + Adaptive Weights")
    print("=" * 60)
    confusion_pairs, raw_cm = compute_confusion_pairs_and_matrix(
        base_model, train_dataset, rare_ids, device=DEVICE, top_k=CONF_TOP_K)
    confusion_weights = compute_adaptive_confusion_weights(
        raw_cm, rare_ids, base_alpha=CONF_BASE_ALPHA)
    save_adaptive_weight_heatmap(confusion_weights, rare_ids, out_dir=OUT_DIR)
    np.save(os.path.join(OUT_DIR, "confusion_weights.npy"), confusion_weights.numpy())
    np.save(os.path.join(OUT_DIR, "raw_confusion_matrix.npy"), raw_cm)
    conf_pairs_serial = {str(k): [int(v) for v in vl]
                         for k, vl in confusion_pairs.items()}
    with open(os.path.join(OUT_DIR, "confusion_pairs.json"), "w") as f:
        json.dump(conf_pairs_serial, f, indent=2)

    # ══════════════════════════════════════════════════════
    # PHASE A→B: Build Dual KG
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print(f"PHASE A→B: Dual KG + Rare Node Amp×{RARE_AMP_FACTOR}")
    print("=" * 60)
    kg_global_path   = os.path.join(OUT_DIR, "kg_global.json")
    kg_minority_path = os.path.join(OUT_DIR, "kg_minority.json")
    if os.path.exists(kg_global_path) and os.path.exists(kg_minority_path):
        print("  Found cached KG — checking confusion edges ...")
        dual_kg = DualKnowledgeGraph.load(
            OUT_DIR, rare_ids=rare_ids, emb_dim=base_model.sent_out_dim)
        if len(dual_kg.g_all.conf_cx_edges) == 0 and len(confusion_pairs) > 0:
            dual_kg = build_dual_knowledge_graph(
                base_model, train_docs, tokenizer, rare_ids,
                confusion_pairs=confusion_pairs,
                confusion_weights=confusion_weights, device=DEVICE)
        else:
            dual_kg.build_confusion_edges(confusion_pairs, confusion_weights)
            dual_kg.save(OUT_DIR)
    else:
        dual_kg = build_dual_knowledge_graph(
            base_model, train_docs, tokenizer, rare_ids,
            confusion_pairs=confusion_pairs,
            confusion_weights=confusion_weights, device=DEVICE)

    # ══════════════════════════════════════════════════════
    # PHASE B: DualKG + DA-IA-SCL + Rare-only Mixup
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: DualKG + Adaptive Edges + DA-IA-SCL + Rare-only Mixup")
    print(f"  Rare amp: G_min×{RARE_AMP_FACTOR}  noise={RARE_AMP_NOISE}")
    print(f"  Mixup: α={MIXUP_ALPHA}  weight={MIXUP_LOSS_WEIGHT}")
    print(f"  SCL weight={SCL_WEIGHT}")
    print(f"  G_all conf edges: {len(dual_kg.g_all.conf_cx_edges)}")
    print(f"  G_min conf edges: {len(dual_kg.g_min.conf_cx_edges)}")
    print("=" * 60)

    dual_retriever = DualKGRetriever(
        dual_kg=dual_kg, label_freqs=label_freqs, rare_ids=rare_ids)

    kg_model = DualKGAugmentedModel(
        base_model        = base_model,
        dual_kg           = dual_kg,
        dual_retriever    = dual_retriever,
        rare_ids          = rare_ids,
        scl_loss_fn       = scl_loss_fn,
        scl_class_weights = scl_class_weights,
        scl_sampler       = scl_sampler)
    total_trainable, _ = count_parameters(kg_model)

    kg_trainer = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids=rare_ids, num_epochs=NUM_EPOCHS_KG)

    # Phase B evaluation
    print("\nPhase B evaluation ...")
    kg_dev_metrics  = kg_trainer.evaluate(dev_dataset,  rare_ids,
                                           split_name="kg_dev",
                                           measure_inference_time=True)
    kg_test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                           split_name="kg_test",
                                           measure_inference_time=True)
    print(f"  [KG+SCL] Dev  Macro-F1:{kg_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1:{kg_dev_metrics['rare_f1']:.4f}")
    print(f"  [KG+SCL] Test Macro-F1:{kg_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1:{kg_test_metrics['rare_f1']:.4f}")

    for split, m in [("kg_dev", kg_dev_metrics), ("kg_test", kg_test_metrics)]:
        with open(os.path.join(OUT_DIR, f"{split}_classification_report.txt"), "w") as f:
            f.write("Phase B: DualKG + DA-IA-SCL + RareMixup\n\n")
            f.write(m["cls_report"])
        save_confusion_matrix(m["cm"], split, rare_labels)
        save_per_class_f1_chart(m["per_class_metrics"], split, rare_labels)

    torch.save({k: v.cpu().clone() for k, v in kg_model.state_dict().items()},
               os.path.join(BEST_MODEL_DIR, "dual_kg_da_ia_scl_mixup_model.bin"))

    # ══════════════════════════════════════════════════════
    # PHASE C+D: PPO  — freeze SCL heads FIRST
    # ══════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print(f"PHASE C+D: PPO  (mode={PHASE_D_MODE})")
    print("  ★ Freezing SCL projection heads before PPO ★")
    print("=" * 60)

    # ── CRITICAL: freeze BEFORE PPOModel is created ──────
    kg_model.freeze_scl_heads()

    ppo_model   = PPOModel(kg_model=kg_model, rare_ids=rare_ids)
    ppo_trainer = PPOTrainer(
        ppo_model=ppo_model, rare_ids=rare_ids,
        dual_retriever=dual_retriever, device=DEVICE)

    ppo_hist_df, ppo_time = ppo_trainer.train(
        train_dataset, dev_dataset, rare_ids=rare_ids, num_epochs=PPO_EPOCHS)

    # PPO Evaluation
    print("\nEvaluating PPO on Dev set ...")
    ppo_dev_metrics = ppo_trainer.evaluate_full(dev_dataset,  rare_ids,
                                                 split_name="ppo_dev")
    print(f"  [PPO] Dev  Macro-F1:{ppo_dev_metrics['macro_f1']:.4f}  "
          f"Rare-F1:{ppo_dev_metrics['rare_f1']:.4f}")

    print("\nEvaluating PPO on Test set ...")
    ppo_test_metrics = ppo_trainer.evaluate_full(test_dataset, rare_ids,
                                                  split_name="ppo_test")
    print(f"  [PPO] Test Macro-F1:{ppo_test_metrics['macro_f1']:.4f}  "
          f"Rare-F1:{ppo_test_metrics['rare_f1']:.4f}")

    for split, m in [("ppo_dev", ppo_dev_metrics), ("ppo_test", ppo_test_metrics)]:
        with open(os.path.join(OUT_DIR, f"{split}_classification_report.txt"), "w") as f:
            f.write(f"Phase C+D: PPO (mode={PHASE_D_MODE})\n\n")
            f.write(m["cls_report"])
        save_confusion_matrix(m["cm"], split, rare_labels)
        save_per_class_f1_chart(m["per_class_metrics"], split, rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in ppo_test_metrics["all_trues"]],
        "pred": [id2label[x] for x in ppo_test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "ppo_test_predictions.csv"), index=False)

    # ── Combined training curves ──────────────────────────
    plot_all_phases(base_hist_df, kg_hist_df, ppo_hist_df)

    # ── JSON summary ──────────────────────────────────────
    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision",
        "macro_recall", "micro_recall", "weighted_recall",
        "rare_precision", "rare_recall", "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT+BiLSTM+MHA+CRF+DualKG+AdaptiveEdge+DA-IA-SCL+RareMixup+PPO",
        "phase_d_mode": PHASE_D_MODE,
        "rare_classes": rare_labels,
        "scl_ppo_isolation": {
            "mechanism": "SCL projection heads (base.scl_proj + fused_scl_proj) "
                         "are frozen via freeze_scl_heads() before PPOModel is "
                         "instantiated. PPO update iterates over detached ctx_out "
                         "tensors — no BERT/SCL forward pass in PPO update step.",
            "benefit": "PPO inherits cleaner SCL-shaped embedding geometry for "
                       "KG-disagreement reward signal, without gradient corruption.",
        },
        "da_ia_scl_config": {
            "temperature": SCL_TEMPERATURE,
            "minority_boost": SCL_MINORITY_BOOST,
            "disc_pos_scale": DISC_POS_SCALE,
            "disc_neg_scale": DISC_NEG_SCALE,
            "disc_compat_thresh": DISC_COMPAT_THRESH,
            "imbal_gamma": SCL_IMBAL_GAMMA,
            "weight": SCL_WEIGHT,
        },
        "rare_mixup_config": {
            "alpha": MIXUP_ALPHA,
            "loss_weight": MIXUP_LOSS_WEIGHT,
            "noise_std": MIXUP_NOISE_STD,
        },
        "kg_config": {
            "rare_amp_factor": RARE_AMP_FACTOR,
            "rare_amp_noise": RARE_AMP_NOISE,
            "conf_base_alpha": CONF_BASE_ALPHA,
            "conf_top_k": CONF_TOP_K,
            "g_all_conf_edges": len(dual_kg.g_all.conf_cx_edges),
            "g_min_conf_edges": len(dual_kg.g_min.conf_cx_edges),
        },
        "ppo_config": {
            "reward_rare_correct": REWARD_RARE_CORRECT,
            "reward_rare_wrong":   REWARD_RARE_WRONG,
            "ppo_clip_eps": PPO_CLIP_EPS,
            "ppo_kl_coef": PPO_KL_COEF,
            "gae_gamma": GAE_GAMMA,
            "gae_lambda": GAE_LAMBDA,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "phase_cd_s": ppo_time,
            "total_s": base_time + kg_time + ppo_time,
        },
        "kg_rag_dev":   {k: kg_dev_metrics[k]   for k in scalar_keys},
        "kg_rag_test":  {k: kg_test_metrics[k]  for k in scalar_keys},
        "ppo_dev":      {k: ppo_dev_metrics[k]  for k in scalar_keys},
        "ppo_test":     {k: ppo_test_metrics[k] for k in scalar_keys},
        "per_class_kg_test":  kg_test_metrics["per_class_metrics"],
        "per_class_ppo_test": ppo_test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_final_table(
        kg_dev_metrics,  kg_test_metrics,
        ppo_dev_metrics, ppo_test_metrics,
        base_time=base_time, kg_time=kg_time, ppo_time=ppo_time,
        total_trainable=total_trainable)

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")
    print(f"   Phase A model  : {BEST_MODEL_DIR}/base_da_ia_scl_model.bin")
    print(f"   Phase B model  : {BEST_MODEL_DIR}/dual_kg_da_ia_scl_mixup_model.bin")
    print(f"   Phase C+D model: {BEST_MODEL_DIR}/ppo_daiscl_model.bin")
    print(f"   Metrics JSON   : {OUT_DIR}/metrics_summary.json")


if __name__ == "__main__":
    main()

Device      : cuda:0
Architecture: InLegalBERT + BiLSTM + MHA + CRF
              → Dual KG-RAG (G_all + G_min×5 amp)
              + Adaptive Confusion Cross-Edges
              + Discourse-Aware Imbalance-Aware SCL (DA-IA-SCL)
              + Rare-only Intra-class Manifold Mixup
              + PPO RL (Phase C+D)

  ══ SCL-PPO isolation: SCL proj heads are FROZEN before PPO ══

  Phase D mode : combined

Loading JSONL files ...
  Train:245 | Dev:30 | Test:50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELI

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-7.
🔥 BERT trainable: layers 8-11 + pooler.

[Base+DA-IA-SCL] Ep 001/60 | loss:276.2616 val:212.5454 | mac_F1:0.0391 rare_F1:0.0000 | t:38.7s ES:0/10
  ✔ New best val_macro_f1=0.0391
[Base+DA-IA-SCL] Ep 002/60 | loss:224.3985 val:168.9527 | mac_F1:0.0879 rare_F1:0.0000 | t:38.2s ES:0/10
  ✔ New best val_macro_f1=0.0879
[Base+DA-IA-SCL] Ep 003/60 | loss:184.7858 val:126.7727 | mac_F1:0.2332 rare_F1:0.0854 | t:37.9s ES:0/10
  ✔ New best val_macro_f1=0.2332
[Base+DA-IA-SCL] Ep 004/60 | loss:147.3977 val:99.8631 | mac_F1:0.2742 rare_F1:0.1099 | t:38.5s ES:0/10
  ✔ New best val_macro_f1=0.2742
[Base+DA-IA-SCL] Ep 005/60 | loss:125.4527 val:90.4490 | mac_F1:0.2827 rare_F1:0.1205 | t:38.2s ES:0/10
  ✔ New best val_macro_f1=0.2827
[Base+DA-IA-SCL] Ep 006/60 | loss:114.9031 val:81.8149 | mac_F1:0.3167 rare_F1:0.1581 | t:38.4s ES:0/10
  ✔ New best val_macro_f1=0.3167
[Base+DA-IA-SCL] Ep 007/60 | loss:104.9962 val:82.8851 | mac_F1:0.3246 rare_F1:0.1732 | t:3